In [27]:
import json
import pandas as pd

In [28]:

with open("data/pl_2024_25_raw.json", "r") as f:
    data = json.load(f)

matches = data["matches"]
print(len(matches))  # should print 380

380


In [29]:
rows = []

for match in matches:
    rows.append({
        "date": match["utcDate"],
        "home_team": match["homeTeam"]["name"],
        "away_team": match["awayTeam"]["name"],
        "home_goals": match["score"]["fullTime"]["home"],
        "away_goals": match["score"]["fullTime"]["away"],
        "status": match["status"]
    })

In [30]:

df = pd.DataFrame(rows)
print(df.head())        # first 5 rows
print(df.shape)         # (380, 6) if everything worked

                   date             home_team                   away_team  \
0  2024-08-16T19:00:00Z  Manchester United FC                   Fulham FC   
1  2024-08-17T11:30:00Z       Ipswich Town FC                Liverpool FC   
2  2024-08-17T14:00:00Z            Arsenal FC  Wolverhampton Wanderers FC   
3  2024-08-17T14:00:00Z            Everton FC   Brighton & Hove Albion FC   
4  2024-08-17T14:00:00Z   Newcastle United FC              Southampton FC   

   home_goals  away_goals    status  
0           1           0  FINISHED  
1           0           2  FINISHED  
2           2           0  FINISHED  
3           0           3  FINISHED  
4           1           0  FINISHED  
(380, 6)


In [31]:
df.to_csv("data/pl_2024_25.csv", index=False)

In [32]:
rows_for_model = []

for _, row in df.iterrows():
    rows_for_model.append({
        "team": row["home_team"],
        "opponent": row["away_team"],
        "is_home": 1,
        "goals": row["home_goals"]
    })
    rows_for_model.append({
        "team": row["away_team"],
        "opponent": row["home_team"],
        "is_home": 0,
        "goals": row["away_goals"]
    })

model_df = pd.DataFrame(rows_for_model)
print(model_df.shape)   
print(model_df.head())

(760, 4)
                   team                    opponent  is_home  goals
0  Manchester United FC                   Fulham FC        1      1
1             Fulham FC        Manchester United FC        0      0
2       Ipswich Town FC                Liverpool FC        1      0
3          Liverpool FC             Ipswich Town FC        0      2
4            Arsenal FC  Wolverhampton Wanderers FC        1      2


In [33]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.glm(
    formula="goals ~ team + opponent + is_home",
    data=model_df,
    family=sm.families.Poisson()
).fit()

attack_coefs = model.params[model.params.index.str.startswith("team[")]
print(attack_coefs.sort_values(ascending=False))

team[T.Liverpool FC]                  3.902428e-01
team[T.Manchester City FC]            2.148923e-01
team[T.Arsenal FC]                    1.627625e-01
team[T.Newcastle United FC]           1.604274e-01
team[T.Brighton & Hove Albion FC]     1.419675e-01
team[T.Brentford FC]                  1.400441e-01
team[T.Tottenham Hotspur FC]          1.168700e-01
team[T.Chelsea FC]                    9.584819e-02
team[T.Aston Villa FC]                4.719787e-03
team[T.Nottingham Forest FC]          5.967449e-16
team[T.Wolverhampton Wanderers FC]   -4.979395e-02
team[T.Fulham FC]                    -6.408871e-02
team[T.Crystal Palace FC]            -1.242143e-01
team[T.West Ham United FC]           -2.172815e-01
team[T.Manchester United FC]         -2.693597e-01
team[T.Everton FC]                   -3.252646e-01
team[T.Ipswich Town FC]              -4.440518e-01
team[T.Leicester City FC]            -5.331870e-01
team[T.Southampton FC]               -7.664275e-01
dtype: float64


In [34]:
def predict_expected_goals(home_team, away_team, model):
    home_row = pd.DataFrame({
        "team": [home_team],
        "opponent": [away_team],
        "is_home": [1]
    })
    away_row = pd.DataFrame({
        "team": [away_team],
        "opponent": [home_team],
        "is_home": [0]
    })
    
    home_lambda = model.predict(home_row).iloc[0]
    away_lambda = model.predict(away_row).iloc[0]
    
    return home_lambda, away_lambda

home_xg, away_xg = predict_expected_goals("Tottenham Hotspur FC", "Southampton FC", model)
print(f"Expected goals — Home: {home_xg:.2f}, Away: {away_xg:.2f}")

Expected goals — Home: 2.64, Away: 0.80


In [35]:
from scipy.stats import poisson
import numpy as np

def match_outcome_probabilities(home_xg, away_xg, max_goals=10):

    home_probs = [poisson.pmf(i, home_xg) for i in range(max_goals + 1)]
    away_probs = [poisson.pmf(j, away_xg) for j in range(max_goals + 1)]
    
    score_matrix = np.outer(home_probs, away_probs)
    
    home_win = np.sum(np.tril(score_matrix, -1))   # home goals > away goals
    draw = np.sum(np.diag(score_matrix))             # home goals == away goals
    away_win = np.sum(np.triu(score_matrix, 1))      # away goals > home goals
    
    return home_win, draw, away_win

home_win, draw, away_win = match_outcome_probabilities(home_xg, away_xg)
print(f"Home win: {home_win:.1%}, Draw: {draw:.1%}, Away win: {away_win:.1%}")

Home win: 76.5%, Draw: 14.5%, Away win: 9.0%
